In [1]:
import pandas as pd
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory

In [2]:
produtos = ['desktop','laptop']

restrições = {
    'desktop':80,
    'laptop':75,
    'horas':300
    
}

informacoes = {
    'desktop':{
      'lucro':600, 
      'horas':2, 
    },
    'laptop':{
      'lucro':900,  
      'horas':3, 
    }
}

In [21]:
model = ConcreteModel()

model.produtos = pyo.Set(initialize=produtos)
# model.restricoes = pyo.Set(initialize=restrições.keys())
model.x = pyo.Var(model.produtos, bounds=(0,None), domain=pyo.NonNegativeIntegers)

def objetivo(model):
    return sum(
        model.x[p]*informacoes[p]['lucro']  for p in model.produtos
    )
model.objetivo = pyo.Objective(rule=objetivo, sense=pyo.maximize)

def restricoes_quantidade(model, p):
    return model.x[p] <= restrições[p]
model.restricao_quantidade = pyo.Constraint(model.produtos, rule=restricoes_quantidade)

def restricao_horas(model):
    return sum(model.x[p] * informacoes[p]['horas'] for p in model.produtos) <= restrições['horas']
model.restricao_horas = pyo.Constraint(rule=restricao_horas)

In [22]:
model.pprint()

1 Set Declarations
    produtos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    2 : {'desktop', 'laptop'}

1 Var Declarations
    x : Size=2, Index=produtos
        Key     : Lower : Value : Upper : Fixed : Stale : Domain
        desktop :     0 :  None :  None : False :  True : NonNegativeIntegers
         laptop :     0 :  None :  None : False :  True : NonNegativeIntegers

1 Objective Declarations
    objetivo : Size=1, Index=None, Active=True
        Key  : Active : Sense    : Expression
        None :   True : maximize : 600*x[desktop] + 900*x[laptop]

2 Constraint Declarations
    restricao_horas : Size=1, Index=None, Active=True
        Key  : Lower : Body                       : Upper : Active
        None :  -Inf : 2*x[desktop] + 3*x[laptop] : 300.0 :   True
    restricao_quantidade : Size=2, Index=produtos, Active=True
        Key     : Lower : Body       : Upper : Active
        desktop :  -Inf : x[des

In [23]:
opt = SolverFactory('cplex')
res = opt.solve(model,tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\DECIV\AppData\Local\Temp\tmp9u19xd3d.cplex.log' open.
CPLEX> Problem 'C:\Users\DECIV\AppData\Local\Temp\tmpzuynvfm_.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\DECIV\AppData\Local\Temp\tmpzuynvfm_.pyomo.lp
Objective sense      : Maximize
Variables            :       2  [General Integer: 2]
Objective nonzeros   :       2
Linear constraints   :       3  [Less: 3]
  Nonzeros           :       4
  RHS nonzeros       :       3

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   : Min   : 600.0000       

In [42]:
for p in model.x:
    print(f"{p} custa R$ {pyo.value(model.x[p]):.2f}")
print(f"Lucro total = R$ {model.objetivo():.2f}")

desktop custa R$ 78.00
laptop custa R$ 48.00
Lucro total = R$ 90000.00
